In [1]:
from datetime import date
from tqdm import tqdm
import shutil
import os

from thumbnails import ThumbnailDef, Thumbnailer, ThumbnailParams
from path_management import PathManager
from bucket_connect import BucketConnector

In [2]:
small = ThumbnailDef(120, 'small')
medium = ThumbnailDef(400, 'medium')
large = ThumbnailDef(1600, 'large')

sizes = [small, large, medium] 
thumbnailer = Thumbnailer(sizes)
paths = PathManager(thumbnailer.name_modifiers)
bucket = BucketConnector('./prod.bucket_config', paths)

In [3]:
client = bucket.get_client()

In [ ]:
def make_all_objects_public(client, bucket_name):
    """
    Iterates through all objects stored one level deep under a GUID
    and applies ACL public-read on each.
    """
    paginator = client.get_paginator("list_objects_v2")

    for page in tqdm(paginator.paginate(Bucket=bucket_name)):
        contents = page.get("Contents", [])
        for obj in tqdm(contents):
            key = obj["Key"]

            # Optionally skip "folder" placeholder objects ending in '/'
            if key.endswith("/"):
                continue

            client.put_object_acl(
                Bucket=bucket_name,
                Key=key,
                ACL="public-read"
            )

In [ ]:
make_all_objects_public(client, 'eschenfeldt-baseball-media')